# depthvo → 3D Gaussian Splatting

End-to-end demo: monocular visual odometry on an image sequence, then a 3DGS model trained
from the VO's keyframe poses and sparse map.

**Run this on the GPU machine** with the 3DGS environment as kernel:

```bash
uv sync                                  # depthvo's own venv (VO, eval)
env/gaussian_splatting.sh                # once: 3DGS env with the CUDA extensions (~10 min)
source env/gaussian_splatting.activate
python -m ipykernel install --user --name gaussian_splatting
jupyter lab notebooks/demo_3dgs.ipynb    # pick the gaussian_splatting kernel
```

Pipeline: `images + intrinsics ─vo→ traj.txt + sparse/0 (COLMAP) ─gs→ gaussians.ply + renders`

In [ ]:
import subprocess, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
GS_REPO = REPO / "third_party" / "gaussian-splatting"
sys.path.insert(0, str(GS_REPO))                       # its COLMAP reader, no extra deps
from scene.colmap_loader import read_points3D_binary, read_extrinsics_binary

DATASET, SEQUENCE = "tum_rgbd", "freiburg1_xyz"       # data/tum/freiburg1_xyz/{rgb,groundtruth.txt}
FRONTEND, BACKEND = "klt", "window"
ITERATIONS = 7000
run_dir = REPO / "outputs" / DATASET / SEQUENCE / f"{FRONTEND}_{BACKEND}"
gs_dir = run_dir / "gs"

def depthvo(*stages, **overrides):
    cmd = [str(REPO / ".venv/bin/python"), "run.py", f"dataset={DATASET}", f"sequence={SEQUENCE}",
           f"frontend={FRONTEND}", f"backend={BACKEND}", f"stages=[{','.join(stages)}]",
           *(f"{k}={v}" for k, v in overrides.items())]
    print(" ".join(cmd[1:]))
    subprocess.run(cmd, cwd=REPO, check=True)
print("run dir:", run_dir)

## 1. Visual odometry

KLT tracks → keyframes → GTSAM sliding-window bundle adjustment → PnP for the frames in between.
Skipped below if the run already exists. To watch it live in a viser window (re-runs the VO, ~20 s):

```python
depthvo("vo", "eval", viewer=True)      # then open http://localhost:8080
```

In [ ]:
if not (run_dir / "sparse/0/points3D.bin").exists():
    depthvo("vo", "eval")
else:
    print("VO output exists, skipping")
print(open(run_dir / "ate.json").read() if (run_dir / "ate.json").exists() else "no ground truth → no ATE")

In [ ]:
from IPython.display import Image, display
if (run_dir / "ate.png").exists():
    display(Image(filename=run_dir / "ate.png", width=1200))

### The same thing as a library, with per-frame reprojection error
`run.py` wraps one class. Driving it yourself gives a `FrameResult` per frame with the reprojection
error of that frame's observations before and after the window solve, so the effect of bundle
adjustment is visible per keyframe (PnP pose in, refined pose out).

In [ ]:
from depthvo import VisualOdometry, Intrinsics, ImageSequence

intr = Intrinsics.load(REPO / "configs/dataset/intrinsics/tum_freiburg1.yaml")
vo = VisualOdometry(intr, frontend=FRONTEND, backend=BACKEND)
for frame in ImageSequence(REPO / "data/tum" / SEQUENCE / "rgb", intr, max_frames=120):
    r = vo.step(frame)
    if r.is_keyframe:
        print(r)
print("frame 50 now:", vo.reprojection_error(50), "| pose:", vo.pose(50)[:3, 3].round(3))

In [ ]:
from IPython.display import Image, display
if (run_dir / "reproj.png").exists():
    display(Image(filename=run_dir / "reproj.png", width=1100))   # written by the eval stage for the full run

### The sparse map that seeds 3DGS
Every keyframe pose and triangulated landmark from the VO, in COLMAP format. 3DGS initialises one
Gaussian per landmark (positions and colours), so the quality of this map is what the next step builds on.

In [ ]:
xyz, rgb, err = read_points3D_binary(run_dir / "sparse/0/points3D.bin")
images = read_extrinsics_binary(run_dir / "sparse/0/images.bin")
print(f"{len(xyz)} landmarks, {len(images)} keyframes, mean reprojection error {err.mean():.2f} px")

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(projection="3d")
ax.scatter(xyz[:, 0], xyz[:, 2], -xyz[:, 1], c=rgb / 255, s=1)       # OpenCV frame: y down → plot -y as up
ax.set_xlabel("x"); ax.set_ylabel("z (forward)"); ax.set_zlabel("-y (up)")
ax.set_title("VO sparse map (camera 0 at the origin)")
plt.show()

## 2. Train 3DGS

`train.py` from the graphdeco repo, run in the 3DGS env on the COLMAP model above. `--eval` holds
out every 8th keyframe as test views, so the renders below are of poses the model never trained on.
The `gs` stage of `run.py` does exactly this; here it's called through `run.py` so the command line is
the same one a script would use.

In [ ]:
ply = gs_dir / f"point_cloud/iteration_{ITERATIONS}/point_cloud.ply"
if not ply.exists():
    depthvo("gs", **{"gs.iterations": ITERATIONS, "gs.python": sys.executable})
else:
    print("3DGS model exists, skipping")
print(open(run_dir / "gs.log").read().splitlines()[-3:])

In [ ]:
from plyfile import PlyData
g = PlyData.read(ply)["vertex"]
opacity = 1 / (1 + np.exp(-g["opacity"]))
print(f"{len(g)} Gaussians (from {len(xyz)} initial landmarks), "
      f"{(opacity > 0.5).sum()} with opacity > 0.5")

## 3. Held-out renders vs. real frames

In [ ]:
from PIL import Image as PILImage
test = gs_dir / "test" / f"ours_{ITERATIONS}"
names = sorted(p.name for p in (test / "renders").iterdir())
show = names[:: max(1, len(names) // 4)][:4]
fig, axes = plt.subplots(2, len(show), figsize=(4 * len(show), 6.4))
for j, n in enumerate(show):
    axes[0, j].imshow(PILImage.open(test / "gt" / n)); axes[0, j].set_title(f"input (held out) {n}", fontsize=9)
    axes[1, j].imshow(PILImage.open(test / "renders" / n)); axes[1, j].set_title("3DGS render", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Looking at the model interactively

The trained model is a standard 3DGS `.ply`. Drag it into [SuperSplat](https://superspl.at/editor)
in a browser, or any 3DGS viewer. Note the world frame is the VO's: camera 0 at the origin, OpenCV
axes (x right, y down, z forward), scale arbitrary (monocular; metric once depth constraints are on).

In [ ]:
print(ply)